# Week 1, Day 2: Data Cleaning
**EMA Crossover ML Project**

## Goals for Today (2 hours):
- ✅ Handle missing values (drop vs impute)
- ✅ Check for duplicate signals
- ✅ Validate timestamp ordering
- ✅ Convert boolean columns properly
- ✅ Fix corrupt/invalid records (UTC issues, rogue candle counts)
- ✅ **Deliverable:** Clean dataset ready for analysis

---

## 🎯 Known Issues to Fix:
1. UTC timestamp format corruption in 2 columns
2. Rogue candle count values (200+ when should be < 100)
3. Unexpected outliers in features

## 1. Setup & Load Yesterday's Data

In [ ]:
# TODO: Import libraries you'll need
# Hint: pandas, numpy, matplotlib, seaborn, supabase


# TODO: Set pandas display options for easier viewing


print("✅ Imports loaded")

In [ ]:
# TODO: Connect to Supabase (copy from yesterday)
# SUPABASE_URL = "..."
# SUPABASE_KEY = "..."



In [ ]:
# TODO: Fetch data from Supabase (copy query from yesterday)
# Hint: supabase.table("signals").select("*").eq("status", "analyzed").execute()


# TODO: Convert to DataFrame


# TODO: Quick check - print shape and first few rows



## 2. Investigate the UTC Timestamp Issue

### 🤔 Research Questions:
- What's the difference between `pd.to_datetime()` with and without `utc=True`?
- What happens when you parse a timestamp that's already UTC?
- How do you check if a datetime column is timezone-aware vs naive?

### 📚 Resources to Google:
- "pandas datetime timezone aware vs naive"
- "pandas to_datetime utc parameter"
- "how to check timezone info pandas"

In [ ]:
# TODO: Examine the problematic timestamp columns
# Columns to check: 'time_of_max_price', 'time_of_min_price', 'checked_at_utc'

# STEP 1: Print the first few values of each timestamp column
# What do they look like? Are they strings? Datetime objects? Timezone-aware?



# STEP 2: Check the data type
# Hint: df['column_name'].dtype



# STEP 3: Check for timezone info
# Hint: df['column_name'].dt.tz (if it's already datetime)




In [ ]:
# TODO: Fix the timestamp columns
# Strategy: Try converting to datetime, handle errors
# Research: Look up pd.to_datetime() parameters - 'errors' parameter is useful!

# Option 1: errors='coerce' - turns bad values into NaT (Not a Time)
# Option 2: errors='raise' - stops and shows you what's wrong
# Option 3: errors='ignore' - leaves bad values as-is

# Which one should you use for learning? (Hint: start with 'raise' to see errors)



# After fixing, validate by checking:
# - How many NaT values did you get?
# - What's the date range now?
# - Do the timestamps make sense?



## 3. Investigate Rogue Candle Counts

### 🤔 Think About:
- If backfill runs every 15 minutes and labels look 24-48 hours ahead...
- How many 15-minute candles are in 24 hours? (Hint: 24 hrs * 4 candles/hr = ?)
- How many in 48 hours?
- So what's a reasonable maximum value?

### 🔍 Investigation Steps:
1. What's the distribution of these values?
2. How many are above the "reasonable" threshold?
3. Are they concentrated in specific time periods? Specific symbols?
4. Decision: Delete these rows or cap the values?

In [ ]:
# TODO: Investigate candles_to_max_price and candles_to_min_price

# STEP 1: Calculate theoretical maximum
# If labeling window is 48 hours, and candles are 15-min each:
max_candles_theoretical = # your calculation here
print(f"Theoretical max candles in 48hrs: {max_candles_theoretical}")


# STEP 2: Check actual distribution
# Use .describe(), .max(), .min(), make a histogram



# STEP 3: Count how many exceed the theoretical max
# Hint: Boolean indexing - df[df['column'] > threshold]




In [ ]:
# TODO: Visualize the problem
# Create a histogram showing the distribution
# Add a vertical line at your theoretical maximum
# Research: plt.axvline() for vertical lines




In [ ]:
# TODO: Decide on a strategy and implement it

# DECISION TIME:
# Option A: Delete rows where candle count > theoretical_max
# Option B: Set outlier values to NaN (and handle later)
# Option C: Keep them (if you think they're valid)

# Which makes sense for ML? Remember: garbage in = garbage out



# After your decision, print how many rows affected



## 4. Missing Values Strategy

### 🎓 Learning Goal: When to DROP vs IMPUTE

#### DROP when:
- Less than 5% of data is missing
- The missing data is in the TARGET variable (you need labels!)
- Missing data appears random (not systematic)
- You have plenty of data left after dropping

#### IMPUTE when:
- More than 5% of data is missing (too much to lose)
- Missing data is in FEATURE columns (predictors)
- Missing data has a pattern (e.g., always missing for specific symbol)
- The feature is important for your model

#### IMPUTATION METHODS:
- **Mean/Median**: For numerical features with normal distribution
- **Mode**: For categorical features
- **Forward/Backward Fill**: For time-series data
- **Zero**: When missing means "absence" (e.g., volume)

### 📚 Google These:
- "pandas fillna methods"
- "sklearn SimpleImputer"
- "when to drop vs impute missing data"

In [ ]:
# TODO: Analyze missing values

# STEP 1: Count missing values per column
# Hint: df.isnull().sum()



# STEP 2: Calculate percentage missing
# Hint: (missing_count / total_rows) * 100



# STEP 3: Visualize (if any missing)
# Create a bar chart showing % missing per column



In [ ]:
# TODO: Handle missing values based on YOUR analysis

# For each column with missing data, decide:
# 1. Is it < 5% missing? → Consider dropping rows
# 2. Is it a label column? → MUST drop (can't train without labels)
# 3. Is it a feature? → Consider imputation

# LABEL COLUMNS (must have these):
label_cols = ['max_move_up_pct', 'max_move_down_pct', 'candles_to_max_price', 'candles_to_min_price']

# Example strategy (you implement):
# df = df.dropna(subset=label_cols)  # Drop rows missing labels
# df['some_feature'].fillna(df['some_feature'].median(), inplace=True)  # Impute feature



# After cleaning, check:
print(f"Rows before: {original_len}")
print(f"Rows after: {len(df)}")
print(f"Rows dropped: {original_len - len(df)}")

## 5. Check for Duplicates

### 🤔 What makes a signal "duplicate"?
- Same `symbol` + same `checked_at_utc` = definitely duplicate
- Same `symbol` + very close timestamps (< 1 min apart) = possible duplicate

### 📚 Research:
- "pandas duplicated() method"
- "pandas drop_duplicates() subset parameter"

In [ ]:
# TODO: Check for exact duplicates

# STEP 1: Check if entire rows are duplicated
# Hint: df.duplicated().sum()



# STEP 2: Check for duplicates based on (symbol, timestamp)
# This is more important - same signal can't occur twice at same time
# Hint: df.duplicated(subset=['symbol', 'checked_at_utc']).sum()



# STEP 3: If found, examine them before dropping
# Look at a few duplicate rows - are they truly identical?



In [ ]:
# TODO: Remove duplicates (if any found)

# Decision: Keep first occurrence or last occurrence?
# Hint: drop_duplicates(subset=[], keep='first' or 'last')



# Verify
print(f"Duplicates remaining: {df.duplicated(subset=['symbol', 'checked_at_utc']).sum()}")

## 6. Validate Timestamp Ordering

### ⚠️ CRITICAL FOR TIME-SERIES ML

Why this matters:
- When you split train/test, old data must be train, new data must be test
- If data is shuffled, you get **lookahead bias** (using future to predict past)
- Your model will look amazing in testing but fail in real trading

### 📚 Google:
- "pandas sort_values"
- "pandas is_monotonic_increasing"

In [ ]:
# TODO: Check if timestamps are sorted

# STEP 1: Check if 'checked_at_utc' is sorted (oldest → newest)
# Hint: df['checked_at_utc'].is_monotonic_increasing



# STEP 2: If not sorted, sort it!
# Hint: df.sort_values(by='checked_at_utc', inplace=True)



# STEP 3: Reset index after sorting (important!)
# Hint: df.reset_index(drop=True, inplace=True)



In [ ]:
# TODO: Validate the sort worked

# Print first and last timestamps


# Check if monotonic increasing



## 7. Boolean Columns Validation

### Boolean columns in your data:
- `price_above_both_emas`
- `htf_4h_bias`
- `htf_1d_bias`
- `btc_trend_bias`

### 🤔 Things to check:
- Are they actually boolean (True/False)?
- Or are they stored as integers (0/1)?
- Or strings ('true'/'false')?
- Any NaN/None values?

### 📚 Research:
- "pandas convert to boolean"
- "pandas astype bool"

In [ ]:
# TODO: Check boolean columns

bool_cols = ['price_above_both_emas', 'htf_4h_bias', 'htf_1d_bias', 'btc_trend_bias']

# STEP 1: Check data type of each
# Hint: df[bool_cols].dtypes



# STEP 2: Check unique values in each
# Hint: df['column'].unique()



# STEP 3: Count any NaN values



In [ ]:
# TODO: Convert to proper boolean if needed

# If stored as 0/1 or 'true'/'false', convert to True/False
# Hint: df['column'].astype(bool) - but be careful with this!
# Research: What happens when you convert 0 to bool? What about 1?



# Handle any NaN values in boolean columns
# Decision: Drop rows or fill with False (or True)?



## 8. Final Validation & Data Quality Report

### Run these checks before saving:

In [ ]:
# TODO: Create a data quality report

print("="*60)
print("📋 DATA QUALITY REPORT - AFTER CLEANING")
print("="*60)

# Total rows
print(f"\nTotal rows: {len(df):,}")

# Date range
# ...

# Missing values check
# ...

# Duplicates check
# ...

# Timestamp ordering check
# ...

# Data type validation
# How many numeric? Boolean? Datetime?


print("\n" + "="*60)

In [ ]:
# TODO: Sanity checks on key columns

# Check: Are all candle counts < 200 (your threshold)?


# Check: Are all timestamps valid (no NaT)?


# Check: Do all signals have labels?


# Check: Are there 5 unique symbols?


print("\n✅ All sanity checks passed!")

## 9. Save Cleaned Dataset

### 💾 Two formats:
- **CSV**: Human-readable, universal
- **Pickle**: Preserves data types, faster to load

Save both! CSV for checking, Pickle for modeling.

In [ ]:
# TODO: Save cleaned data

# Save as CSV
# Hint: df.to_csv('filename.csv', index=False)


# Save as Pickle (preserves datetime, boolean types)
# Hint: df.to_pickle('filename.pkl')


print("✅ Cleaned data saved!")

## 10. Document Your Decisions

### 📝 Write down what you did:

**Issues Found:**
1. UTC timestamp corruption: [describe what you found]
2. Rogue candle counts: [how many? which symbols?]
3. Missing values: [which columns? how many?]
4. Duplicates: [how many found?]

**Actions Taken:**
1. Timestamps: [how did you fix them?]
2. Candle counts: [did you delete or cap?]
3. Missing values: [dropped or imputed? which method?]
4. Duplicates: [kept first or last?]

**Final Stats:**
- Original rows: [number]
- Final rows: [number]
- Rows removed: [number] ([percentage]%)

**Ready for next step?** YES / NO

---

✅ Day 2 Complete!

**Tomorrow (Day 3):** Exploratory Data Analysis (EDA)
- Feature distributions
- Correlation analysis
- Signal pattern analysis